## Project objective

            Build a small binary classifier and investigate the mathematics
            behind its predictions.

            You will examine:
            - feature scaling;
            - logistic probabilities;
            - binary cross-entropy;
            - regularization;
            - classification thresholds;
            - performance on held-out data.

            The data are synthetic. This exercise is for learning and is
            not a real credit assessment system.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
)

X, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=5,
    n_redundant=2,
    weights=[0.85, 0.15],
    class_sep=1.0,
    random_state=42,
)

feature_names = [
    "income",
    "debt_ratio",
    "payment_history",
    "credit_utilization",
    "account_age",
    "employment_stability",
    "recent_inquiries",
    "savings",
]

X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
    ),
)

model.fit(X_train, y_train)

probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

print("Positive-class prevalence:", round(y.mean(), 3))
print("Accuracy:", round(
    accuracy_score(y_test, predictions), 3
))
print("Precision:", round(
    precision_score(y_test, predictions, zero_division=0), 3
))
print("Recall:", round(
    recall_score(y_test, predictions, zero_division=0), 3
))
print("ROC-AUC:", round(
    roc_auc_score(y_test, probabilities), 3
))
print("Log loss:", round(log_loss(y_test, probabilities), 3))
print("\\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))

Positive-class prevalence: 0.154
Accuracy: 0.83
Precision: 0.3
Recall: 0.078
ROC-AUC: 0.78
Log loss: 0.369
\nConfusion matrix:
[[409  14]
 [ 71   6]]


## 2. Examine the threshold

A false negative and a false positive can have different
consequences. The threshold should therefore reflect the
purpose of the model and the costs of errors.

Do not select a threshold using the final test labels if you
want an unbiased final performance estimate. In a real project,
tune it on validation data and evaluate once on the test set.